In [4]:
# ============================================================
# V6 — FINAL ONLINE ADAPTIVE LLM ROUTER
# ============================================================
#
# V5:
#   Contextual epsilon-greedy bandit
#
# V6:
#   V5 state
#      ↓
#   DynamoDB persistent state
#      ↓
#   online request
#      ↓
#   explore / exploit
#      ↓
#   model outcome
#      ↓
#   reward
#      ↓
#   DynamoDB update
#      ↓
#   next request
#
# AWS:
#   SageMaker + S3 + DynamoDB + CloudWatch
#
# IMPORTANT:
#   Model outcomes are still simulated.
#   DynamoDB persistence is real.
# ============================================================

import os
import json
import time
import boto3
import numpy as np
import pandas as pd

from decimal import Decimal
from botocore.exceptions import ClientError


# ============================================================
# 1. CONFIGURATION
# ============================================================

PROJECT_ROOT = (
    "/home/sagemaker-user/"
    "adaptive-llm-routing"
)

V5_DIR = os.path.join(
    PROJECT_ROOT,
    "notebooks",
    "data",
    "evaluation",
    "v5"
)

V6_DIR = os.path.join(
    PROJECT_ROOT,
    "notebooks",
    "data",
    "evaluation",
    "v6"
)

os.makedirs(
    V6_DIR,
    exist_ok=True
)


RANDOM_SEED = 42

ONLINE_REQUESTS = 300

INITIAL_EPSILON = 0.05

MIN_EPSILON = 0.01

EPSILON_DECAY = 0.995

COST_PENALTY = 0.10

LATENCY_PENALTY = 0.10

QUALITY_VIOLATION_PENALTY = 1.00

SUCCESS_THRESHOLD = 0.95

RESTART_STEP = 150


MODELS = [
    "small_model",
    "medium_model",
    "large_model"
]

COMPLEXITIES = [
    "simple",
    "medium",
    "complex"
]

QUALITY_REQUIREMENT = {
    "simple": 0.80,
    "medium": 0.90,
    "complex": 0.95
}


print("=" * 100)
print("V6 — FINAL ONLINE ADAPTIVE LLM ROUTER")
print("=" * 100)


# ============================================================
# 2. DYNAMODB NUMBER HELPER
# ============================================================

def ddb_number(value):
    """
    DynamoDB requires Decimal instead of Python float.
    """
    return Decimal(str(value))


# ============================================================
# 3. AWS SESSION
# ============================================================

session = boto3.Session()

region = session.region_name

if region is None:
    raise RuntimeError(
        "Could not determine AWS region."
    )

sts = session.client("sts")

account_id = (
    sts.get_caller_identity()["Account"]
)

s3 = session.client(
    "s3",
    region_name=region
)

dynamodb = session.resource(
    "dynamodb",
    region_name=region
)

dynamodb_client = session.client(
    "dynamodb",
    region_name=region
)

cloudwatch = session.client(
    "cloudwatch",
    region_name=region
)

bucket_name = (
    f"adaptive-llm-router-"
    f"{account_id}-"
    f"{region}"
).lower()

S3_V5_PREFIX = (
    "adaptive-llm-router/v5"
)

S3_V6_PREFIX = (
    "adaptive-llm-router/v6"
)

DDB_TABLE_NAME = (
    f"adaptive-llm-router-v6-state-"
    f"{account_id}"
)

print("\nAWS region:")
print(region)

print("\nAWS account:")
print(account_id)

print("\nS3 bucket:")
print(bucket_name)

print("\nDynamoDB table:")
print(DDB_TABLE_NAME)


# ============================================================
# 4. LOAD V5 BANDIT STATE FROM S3
# ============================================================

V5_STATE_KEY = (
    S3_V5_PREFIX +
    "/v5_bandit_state.csv"
)

V5_STATE_LOCAL = os.path.join(
    V6_DIR,
    "v5_bandit_state_source.csv"
)

try:

    s3.download_file(
        bucket_name,
        V5_STATE_KEY,
        V5_STATE_LOCAL
    )

    print(
        "\nDownloaded V5 state from S3:"
    )

    print(
        f"s3://{bucket_name}/{V5_STATE_KEY}"
    )

except ClientError as e:

    print(
        "\nCould not download V5 state from S3."
    )

    local_fallback = os.path.join(
        V5_DIR,
        "v5_bandit_state.csv"
    )

    if not os.path.exists(local_fallback):

        raise FileNotFoundError(
            "V5 bandit state could not be "
            "found in S3 or locally."
        ) from e

    V5_STATE_LOCAL = local_fallback

    print(
        "Using local V5 state."
    )


v5_state_df = pd.read_csv(
    V5_STATE_LOCAL
)

required_v5_columns = {
    "complexity",
    "model",
    "estimated_reward",
    "count"
}

missing_columns = (
    required_v5_columns
    -
    set(v5_state_df.columns)
)

if missing_columns:

    raise ValueError(
        f"V5 state is missing: "
        f"{missing_columns}"
    )

print(
    "\nV5 state records:",
    len(v5_state_df)
)


# ============================================================
# 5. CREATE / LOAD DYNAMODB TABLE
# ============================================================

table = dynamodb.Table(
    DDB_TABLE_NAME
)

try:

    table.load()

    print(
        "\nDynamoDB table already exists."
    )

except dynamodb_client.exceptions.ResourceNotFoundException:

    print(
        "\nDynamoDB table does not exist."
    )

    print(
        "Creating table..."
    )

    dynamodb_client.create_table(

        TableName=DDB_TABLE_NAME,

        KeySchema=[
            {
                "AttributeName": "state_key",
                "KeyType": "HASH"
            }
        ],

        AttributeDefinitions=[
            {
                "AttributeName": "state_key",
                "AttributeType": "S"
            }
        ],

        BillingMode="PAY_PER_REQUEST"
    )

    waiter = dynamodb_client.get_waiter(
        "table_exists"
    )

    waiter.wait(
        TableName=DDB_TABLE_NAME
    )

    print(
        "DynamoDB table created."
    )


# ============================================================
# 6. SEED DYNAMODB FROM V5
# ============================================================
#
# Every complexity/model pair gets persistent state.
# ============================================================

for _, row in v5_state_df.iterrows():

    complexity = str(
        row["complexity"]
    )

    model_name = str(
        row["model"]
    )

    state_key = (
        f"STATE#{complexity}#{model_name}"
    )

    table.put_item(
        Item={

            "state_key":
                state_key,

            "complexity":
                complexity,

            "model":
                model_name,

            "estimated_reward":
                ddb_number(
                    row[
                        "estimated_reward"
                    ]
                ),

            "count":
                int(
                    row["count"]
                ),

            "updated_at":
                ddb_number(
                    time.time()
                )
        }
    )


# Router config

table.put_item(
    Item={

        "state_key":
            "CONFIG",

        "epsilon":
            ddb_number(
                INITIAL_EPSILON
            ),

        "step":
            0,

        "updated_at":
            ddb_number(
                time.time()
            )
    }
)

print(
    "\nV6 DynamoDB state initialized from V5."
)


# ============================================================
# 7. SIMULATED MODEL PERFORMANCE
# ============================================================
#
# These represent actual model outcomes in the experiment.
# Later this section can be replaced with real Bedrock calls.
# ============================================================

simulated_performance = {

    "simple": {

        "small_model": {
            "quality": 0.91,
            "latency_ms": 190,
            "cost": 0.09
        },

        "medium_model": {
            "quality": 0.94,
            "latency_ms": 480,
            "cost": 0.29
        },

        "large_model": {
            "quality": 0.98,
            "latency_ms": 1150,
            "cost": 0.98
        }
    },

    "medium": {

        "small_model": {
            "quality": 0.76,
            "latency_ms": 210,
            "cost": 0.10
        },

        "medium_model": {
            "quality": 0.91,
            "latency_ms": 510,
            "cost": 0.31
        },

        "large_model": {
            "quality": 0.97,
            "latency_ms": 1190,
            "cost": 1.00
        }
    },

    "complex": {

        "small_model": {
            "quality": 0.55,
            "latency_ms": 220,
            "cost": 0.11
        },

        "medium_model": {
            "quality": 0.80,
            "latency_ms": 530,
            "cost": 0.32
        },

        "large_model": {
            "quality": 0.96,
            "latency_ms": 1220,
            "cost": 1.03
        }
    }
}


# ============================================================
# 8. GLOBAL REWARD NORMALIZATION
# ============================================================

all_costs = []
all_latencies = []

for complexity in COMPLEXITIES:

    for model_name in MODELS:

        profile = (
            simulated_performance[
                complexity
            ][
                model_name
            ]
        )

        all_costs.append(
            profile["cost"]
        )

        all_latencies.append(
            profile["latency_ms"]
        )

COST_MIN = min(all_costs)
COST_MAX = max(all_costs)

LATENCY_MIN = min(all_latencies)
LATENCY_MAX = max(all_latencies)


def normalize(
    value,
    minimum,
    maximum
):

    if maximum == minimum:
        return 0.0

    return (
        value - minimum
    ) / (
        maximum - minimum
    )


def calculate_reward(
    complexity,
    quality,
    latency_ms,
    cost
):

    required_quality = (
        QUALITY_REQUIREMENT[
            complexity
        ]
    )

    normalized_cost = normalize(
        cost,
        COST_MIN,
        COST_MAX
    )

    normalized_latency = normalize(
        latency_ms,
        LATENCY_MIN,
        LATENCY_MAX
    )

    quality_violation = max(
        0.0,
        required_quality - quality
    )

    reward = (

        quality

        - COST_PENALTY
        * normalized_cost

        - LATENCY_PENALTY
        * normalized_latency

        - QUALITY_VIOLATION_PENALTY
        * quality_violation
    )

    return float(
        np.clip(
            reward,
            -1.0,
            1.0
        )
    )


# ============================================================
# 9. DYNAMODB STATE ACCESS
# ============================================================

def get_state(
    complexity,
    model_name
):

    state_key = (
        f"STATE#{complexity}#{model_name}"
    )

    response = table.get_item(
        Key={
            "state_key": state_key
        }
    )

    item = response.get(
        "Item"
    )

    if item is None:

        raise RuntimeError(
            f"Missing DynamoDB state: "
            f"{state_key}"
        )

    return {

        "estimated_reward":
            float(
                item[
                    "estimated_reward"
                ]
            ),

        "count":
            int(
                item["count"]
            )
    }


def get_context_state(
    complexity
):

    state = {}

    for model_name in MODELS:

        state[
            model_name
        ] = get_state(
            complexity,
            model_name
        )

    return state


def get_epsilon():

    response = table.get_item(
        Key={
            "state_key": "CONFIG"
        }
    )

    item = response.get(
        "Item"
    )

    if item is None:
        return INITIAL_EPSILON

    return float(
        item["epsilon"]
    )


def update_config(
    epsilon,
    step
):

    table.put_item(
        Item={

            "state_key":
                "CONFIG",

            "epsilon":
                ddb_number(
                    epsilon
                ),

            "step":
                int(step),

            "updated_at":
                ddb_number(
                    time.time()
                )
        }
    )


def update_state(
    complexity,
    model_name,
    reward
):

    state = get_state(
        complexity,
        model_name
    )

    old_count = (
        state["count"]
    )

    old_reward = (
        state["estimated_reward"]
    )

    new_count = (
        old_count + 1
    )

    new_reward = (

        old_reward

        +

        (
            reward
            -
            old_reward
        )
        /
        new_count
    )

    state_key = (
        f"STATE#{complexity}#{model_name}"
    )

    table.update_item(

        Key={
            "state_key":
                state_key
        },

        UpdateExpression=(
            "SET "
            "estimated_reward = :r, "
            "#cnt = :c, "
            "updated_at = :t"
        ),

        ExpressionAttributeNames={
            "#cnt":
                "count"
        },

        ExpressionAttributeValues={

            ":r":
                ddb_number(
                    new_reward
                ),

            ":c":
                int(
                    new_count
                ),

            ":t":
                ddb_number(
                    time.time()
                )
        }
    )

    return (
        new_reward,
        new_count
    )


# ============================================================
# 10. PERSISTENT BANDIT
# ============================================================

class PersistentBandit:

    def __init__(
        self,
        seed
    ):

        self.rng = (
            np.random.default_rng(
                seed
            )
        )

    def select(
        self,
        complexity
    ):

        epsilon = get_epsilon()

        state = get_context_state(
            complexity
        )

        values = {

            model_name:
                state[
                    model_name
                ][
                    "estimated_reward"
                ]

            for model_name in MODELS
        }

        if (
            self.rng.random()
            <
            epsilon
        ):

            selected = (
                self.rng.choice(
                    MODELS
                )
            )

            exploration = True

        else:

            max_reward = max(
                values.values()
            )

            best_models = [

                model

                for model, value
                in values.items()

                if value == max_reward
            ]

            selected = (
                self.rng.choice(
                    best_models
                )
            )

            exploration = False

        return (
            selected,
            exploration,
            epsilon,
            values
        )

    def update(
        self,
        complexity,
        model_name,
        reward,
        step
    ):

        (
            new_reward,
            new_count
        ) = update_state(

            complexity,
            model_name,
            reward
        )

        current_epsilon = (
            get_epsilon()
        )

        new_epsilon = max(
            MIN_EPSILON,
            current_epsilon
            * EPSILON_DECAY
        )

        update_config(
            new_epsilon,
            step
        )

        return (
            new_reward,
            new_count,
            new_epsilon
        )


# ============================================================
# 11. LOAD ACTUAL V1 LABEL FILE
# ============================================================
#
# Your file is:
#
# [
#   {
#     "instruction": "...",
#     "complexity": "simple",
#     "confidence": ...
#   },
#   ...
# ]
#
# It is NOT query -> complexity JSON.
# ============================================================

EVAL_PATH = os.path.join(
    PROJECT_ROOT,
    "notebooks",
    "data",
    "evaluation",
    "v1_complexity_labels.json"
)

if not os.path.exists(
    EVAL_PATH
):

    raise FileNotFoundError(
        EVAL_PATH
    )


with open(
    EVAL_PATH,
    "r",
    encoding="utf-8"
) as f:

    raw_labels = json.load(
        f
    )


records = []


if isinstance(
    raw_labels,
    list
):

    for item in raw_labels:

        if not isinstance(
            item,
            dict
        ):
            continue

        query = item.get(
            "instruction",
            item.get(
                "query",
                item.get(
                    "question",
                    item.get(
                        "prompt",
                        ""
                    )
                )
            )
        )

        complexity = item.get(
            "complexity",
            item.get(
                "label",
                ""
            )
        )

        if not query:
            continue

        complexity = (
            str(complexity)
            .lower()
            .strip()
        )

        if complexity not in COMPLEXITIES:
            continue

        records.append({

            "query":
                str(query).strip(),

            "complexity":
                complexity
        })


elif isinstance(
    raw_labels,
    dict
):

    for query, complexity in (
        raw_labels.items()
    ):

        complexity = (
            str(complexity)
            .lower()
            .strip()
        )

        if complexity not in COMPLEXITIES:
            continue

        records.append({

            "query":
                str(query).strip(),

            "complexity":
                complexity
        })


else:

    raise ValueError(
        "Unsupported V1 label JSON format."
    )


online_source = pd.DataFrame(
    records
)


if len(online_source) == 0:

    raise ValueError(
        "No valid queries found."
    )


online_source = online_source[
    online_source[
        "query"
    ].str.len() > 0
].reset_index(
    drop=True
)


print("\n")
print("=" * 100)
print("ONLINE QUERY SOURCE")
print("=" * 100)

print(
    "Available queries:",
    len(online_source)
)

print(
    "\nComplexity distribution:"
)

print(
    online_source[
        "complexity"
    ].value_counts()
)


# ============================================================
# 12. BUILD ONLINE REQUEST STREAM
# ============================================================

stream_rng = np.random.default_rng(
    2026
)

online_rows = []

per_context = (
    ONLINE_REQUESTS
    //
    len(COMPLEXITIES)
)


for complexity in COMPLEXITIES:

    subset = online_source[
        online_source[
            "complexity"
        ]
        ==
        complexity
    ]

    if len(subset) == 0:

        raise ValueError(
            f"No queries for "
            f"{complexity}."
        )

    indices = stream_rng.choice(

        len(subset),

        size=per_context,

        replace=True
    )

    selected_rows = (
        subset.iloc[
            indices
        ]
    )

    for _, selected_row in (
        selected_rows.iterrows()
    ):

        online_rows.append({

            "query":
                selected_row[
                    "query"
                ],

            "complexity":
                selected_row[
                    "complexity"
                ]
        })


while len(
    online_rows
) < ONLINE_REQUESTS:

    selected = (
        online_source.iloc[
            stream_rng.integers(
                len(
                    online_source
                )
            )
        ]
    )

    online_rows.append({

        "query":
            selected["query"],

        "complexity":
            selected["complexity"]
    })


online_df = pd.DataFrame(
    online_rows
)


print("\n")
print("=" * 100)
print("V6 ONLINE STREAM")
print("=" * 100)

print(
    "Requests:",
    len(online_df)
)

print(
    "\nComplexity distribution:"
)

print(
    online_df[
        "complexity"
    ].value_counts()
)


# ============================================================
# 13. V5 FIXED BASELINE
# ============================================================

v5_state_lookup = {}


for _, row in (
    v5_state_df.iterrows()
):

    complexity = str(
        row["complexity"]
    )

    model_name = str(
        row["model"]
    )

    if (
        complexity
        not in
        v5_state_lookup
    ):

        v5_state_lookup[
            complexity
        ] = {}

    v5_state_lookup[
        complexity
    ][
        model_name
    ] = float(
        row[
            "estimated_reward"
        ]
    )


def v5_fixed_route(
    complexity
):

    return max(

        v5_state_lookup[
            complexity
        ],

        key=
        v5_state_lookup[
            complexity
        ].get
    )


# ============================================================
# 14. RUN V5 FIXED BASELINE
# ============================================================

baseline_rng = np.random.default_rng(
    500
)

v5_online_results = []


for step, (_, row) in enumerate(
    online_df.iterrows(),
    start=1
):

    complexity = row[
        "complexity"
    ]

    selected_model = (
        v5_fixed_route(
            complexity
        )
    )

    base = (
        simulated_performance[
            complexity
        ][
            selected_model
        ]
    )

    quality = float(
        np.clip(
            baseline_rng.normal(
                base["quality"],
                0.04
            ),
            0.0,
            1.0
        )
    )

    latency = float(
        max(
            1.0,
            baseline_rng.normal(
                base["latency_ms"],
                base["latency_ms"]
                * 0.08
            )
        )
    )

    cost = float(
        max(
            0.001,
            baseline_rng.normal(
                base["cost"],
                base["cost"]
                * 0.05
            )
        )
    )

    success_probability = (
        0.95 +
        0.05 * quality
    )

    success = int(
        baseline_rng.random()
        <
        success_probability
    )

    reward = calculate_reward(
        complexity,
        quality,
        latency,
        cost
    )

    v5_online_results.append({

        "step":
            step,

        "complexity":
            complexity,

        "selected_model":
            selected_model,

        "quality":
            quality,

        "latency_ms":
            latency,

        "cost":
            cost,

        "success":
            success,

        "reward":
            reward
    })


v5_online_df = pd.DataFrame(
    v5_online_results
)


# ============================================================
# 15. RUN V6 ONLINE ADAPTATION
# ============================================================

v6_rng = np.random.default_rng(
    600
)

bandit = PersistentBandit(
    seed=600
)

v6_results = []

cumulative_reward = 0.0

exploration_count = 0

restart_recovered = False


for step, (_, row) in enumerate(
    online_df.iterrows(),
    start=1
):

    query = row[
        "query"
    ]

    complexity = row[
        "complexity"
    ]


    # --------------------------------------------------------
    # Simulate a process restart at step 151.
    # The new bandit object reloads everything from DynamoDB.
    # --------------------------------------------------------

    if (
        step
        ==
        RESTART_STEP + 1
    ):

        bandit = PersistentBandit(
            seed=601
        )

        restart_recovered = True


    # --------------------------------------------------------
    # Select using persistent state
    # --------------------------------------------------------

    (
        selected_model,
        exploration,
        epsilon,
        estimates
    ) = bandit.select(
        complexity
    )


    if exploration:

        exploration_count += 1


    # --------------------------------------------------------
    # Simulated model outcome
    # --------------------------------------------------------

    base = (
        simulated_performance[
            complexity
        ][
            selected_model
        ]
    )


    quality = float(
        np.clip(
            v6_rng.normal(
                base["quality"],
                0.04
            ),
            0.0,
            1.0
        )
    )


    latency = float(
        max(
            1.0,
            v6_rng.normal(
                base["latency_ms"],
                base["latency_ms"]
                * 0.08
            )
        )
    )


    cost = float(
        max(
            0.001,
            v6_rng.normal(
                base["cost"],
                base["cost"]
                * 0.05
            )
        )
    )


    success_probability = (
        0.95 +
        0.05 * quality
    )


    success = int(
        v6_rng.random()
        <
        success_probability
    )


    # --------------------------------------------------------
    # Reward
    # --------------------------------------------------------

    reward = calculate_reward(

        complexity,

        quality,

        latency,

        cost
    )


    cumulative_reward += reward


    # --------------------------------------------------------
    # Persist updated state
    # --------------------------------------------------------

    (
        updated_reward,
        updated_count,
        new_epsilon
    ) = bandit.update(

        complexity,

        selected_model,

        reward,

        step
    )


    v6_results.append({

        "step":
            step,

        "query":
            query,

        "complexity":
            complexity,

        "selected_model":
            selected_model,

        "exploration":
            exploration,

        "epsilon":
            epsilon,

        "quality":
            quality,

        "latency_ms":
            latency,

        "cost":
            cost,

        "success":
            success,

        "reward":
            reward,

        "cumulative_reward":
            cumulative_reward,

        "updated_estimated_reward":
            updated_reward,

        "updated_count":
            updated_count,

        "restart_recovered":
            (
                restart_recovered
                and
                step >
                RESTART_STEP
            )
    })


v6_results_df = pd.DataFrame(
    v6_results
)


# ============================================================
# 16. V5 METRICS
# ============================================================

v5_quality = (
    v5_online_df[
        "quality"
    ].mean()
)

v5_latency = (
    v5_online_df[
        "latency_ms"
    ].mean()
)

v5_cost = (
    v5_online_df[
        "cost"
    ].mean()
)

v5_success = (
    v5_online_df[
        "success"
    ].mean()
)

v5_reward = (
    v5_online_df[
        "reward"
    ].mean()
)


# ============================================================
# 17. V6 METRICS
# ============================================================

v6_quality = (
    v6_results_df[
        "quality"
    ].mean()
)

v6_latency = (
    v6_results_df[
        "latency_ms"
    ].mean()
)

v6_cost = (
    v6_results_df[
        "cost"
    ].mean()
)

v6_success = (
    v6_results_df[
        "success"
    ].mean()
)

v6_reward = (
    v6_results_df[
        "reward"
    ].mean()
)

v6_exploration_rate = (
    v6_results_df[
        "exploration"
    ].mean()
)

v6_final_epsilon = (
    v6_results_df[
        "epsilon"
    ].iloc[-1]
)


# ============================================================
# 18. V5 VS V6
# ============================================================

comparison_df = pd.DataFrame({

    "metric": [

        "average_quality",

        "average_latency_ms",

        "average_cost",

        "success_rate",

        "average_reward"
    ],

    "V5": [

        v5_quality,

        v5_latency,

        v5_cost,

        v5_success,

        v5_reward
    ],

    "V6": [

        v6_quality,

        v6_latency,

        v6_cost,

        v6_success,

        v6_reward
    ]
})


comparison_df[
    "delta_V6_minus_V5"
] = (
    comparison_df["V6"]
    -
    comparison_df["V5"]
).round(4)


comparison_df["V5"] = (
    comparison_df["V5"]
    .round(4)
)

comparison_df["V6"] = (
    comparison_df["V6"]
    .round(4)
)


print("\n")
print("=" * 100)
print("V5 VS V6")
print("=" * 100)

print(
    comparison_df.to_string(
        index=False
    )
)


# ============================================================
# 19. MODEL USAGE
# ============================================================

v5_usage = (
    v5_online_df[
        "selected_model"
    ]
    .value_counts()
    .to_frame(
        "requests"
    )
)

v5_usage["percentage"] = (
    v5_usage["requests"]
    /
    len(v5_online_df)
    *
    100
).round(2)


v6_usage = (
    v6_results_df[
        "selected_model"
    ]
    .value_counts()
    .to_frame(
        "requests"
    )
)

v6_usage["percentage"] = (
    v6_usage["requests"]
    /
    len(v6_results_df)
    *
    100
).round(2)


print("\n")
print("=" * 100)
print("V5 MODEL USAGE")
print("=" * 100)

print(v5_usage)


print("\n")
print("=" * 100)
print("V6 MODEL USAGE")
print("=" * 100)

print(v6_usage)


# ============================================================
# 20. READ FINAL DYNAMODB STATE
# ============================================================

final_state_rows = []


for complexity in COMPLEXITIES:

    for model_name in MODELS:

        state = get_state(
            complexity,
            model_name
        )

        final_state_rows.append({

            "complexity":
                complexity,

            "model":
                model_name,

            "estimated_reward":
                state[
                    "estimated_reward"
                ],

            "count":
                state[
                    "count"
                ]
        })


final_state_df = pd.DataFrame(
    final_state_rows
)

final_state_df[
    "estimated_reward"
] = (
    final_state_df[
        "estimated_reward"
    ]
    .round(6)
)


print("\n")
print("=" * 100)
print("FINAL DYNAMODB STATE")
print("=" * 100)

print(
    final_state_df.to_string(
        index=False
    )
)


# ============================================================
# 21. SAVE LOCAL ARTIFACTS
# ============================================================

v6_results_path = os.path.join(
    V6_DIR,
    "v6_online_results.csv"
)

v6_results_df.to_csv(
    v6_results_path,
    index=False
)


v5_baseline_path = os.path.join(
    V6_DIR,
    "v5_online_baseline.csv"
)

v5_online_df.to_csv(
    v5_baseline_path,
    index=False
)


comparison_path = os.path.join(
    V6_DIR,
    "v5_vs_v6_comparison.csv"
)

comparison_df.to_csv(
    comparison_path,
    index=False
)


state_path = os.path.join(
    V6_DIR,
    "v6_dynamodb_state_snapshot.csv"
)

final_state_df.to_csv(
    state_path,
    index=False
)


usage_v5_path = os.path.join(
    V6_DIR,
    "v5_model_usage.csv"
)

v5_usage.to_csv(
    usage_v5_path
)


usage_v6_path = os.path.join(
    V6_DIR,
    "v6_model_usage.csv"
)

v6_usage.to_csv(
    usage_v6_path
)


summary_df = pd.DataFrame({

    "metric": [

        "online_requests",

        "v5_average_quality",

        "v6_average_quality",

        "v5_average_latency_ms",

        "v6_average_latency_ms",

        "v5_average_cost",

        "v6_average_cost",

        "v5_success_rate",

        "v6_success_rate",

        "v5_average_reward",

        "v6_average_reward",

        "v6_exploration_rate",

        "v6_final_epsilon",

        "restart_recovered"
    ],

    "value": [

        len(
            online_df
        ),

        v5_quality,

        v6_quality,

        v5_latency,

        v6_latency,

        v5_cost,

        v6_cost,

        v5_success,

        v6_success,

        v5_reward,

        v6_reward,

        v6_exploration_rate,

        v6_final_epsilon,

        int(
            restart_recovered
        )
    ]
})


summary_path = os.path.join(
    V6_DIR,
    "v6_summary.csv"
)

summary_df.to_csv(
    summary_path,
    index=False
)


# ============================================================
# 22. SAVE V6 CONFIG
# ============================================================

config = {

    "version":
        "V6",

    "algorithm":
        "persistent_online_epsilon_greedy",

    "context":
        "query_complexity",

    "state_store":
        "DynamoDB",

    "historical_store":
        "S3",

    "monitoring":
        "CloudWatch",

    "online_requests":
        ONLINE_REQUESTS,

    "initial_epsilon":
        INITIAL_EPSILON,

    "minimum_epsilon":
        MIN_EPSILON,

    "epsilon_decay":
        EPSILON_DECAY,

    "restart_step":
        RESTART_STEP,

    "dynamodb_table":
        DDB_TABLE_NAME,

    "s3_bucket":
        bucket_name,

    "s3_prefix":
        S3_V6_PREFIX,

    "cloudwatch_namespace":
        "AdaptiveLLMRouter/V6",

    "simulation":
        True
}


config_path = os.path.join(
    V6_DIR,
    "v6_config.json"
)


with open(
    config_path,
    "w"
) as f:

    json.dump(
        config,
        f,
        indent=2
    )


# ============================================================
# 23. UPLOAD V6 ARTIFACTS TO S3
# ============================================================

v6_artifacts = [

    "v6_online_results.csv",

    "v5_online_baseline.csv",

    "v5_vs_v6_comparison.csv",

    "v6_dynamodb_state_snapshot.csv",

    "v5_model_usage.csv",

    "v6_model_usage.csv",

    "v6_summary.csv",

    "v6_config.json"
]


print("\n")
print("=" * 100)
print("UPLOADING V6 ARTIFACTS TO S3")
print("=" * 100)


upload_failures = []


for filename in v6_artifacts:

    local_path = os.path.join(
        V6_DIR,
        filename
    )

    s3_key = (
        S3_V6_PREFIX
        +
        "/"
        +
        filename
    )

    try:

        s3.upload_file(
            local_path,
            bucket_name,
            s3_key
        )

        print(
            f"Uploaded: "
            f"s3://{bucket_name}/"
            f"{s3_key}"
        )

    except ClientError as e:

        upload_failures.append(
            filename
        )

        print(
            f"UPLOAD FAILED: "
            f"{filename}"
        )

        print(e)


# ============================================================
# 24. CLOUDWATCH METRICS
# ============================================================

metric_data = [

    {
        "MetricName":
            "AverageQuality",

        "Value":
            float(v6_quality),

        "Unit":
            "None",

        "Dimensions": [

            {
                "Name":
                    "Version",

                "Value":
                    "V6"
            }
        ]
    },

    {
        "MetricName":
            "AverageReward",

        "Value":
            float(v6_reward),

        "Unit":
            "None",

        "Dimensions": [

            {
                "Name":
                    "Version",

                "Value":
                    "V6"
            }
        ]
    },

    {
        "MetricName":
            "AverageLatencyMs",

        "Value":
            float(v6_latency),

        "Unit":
            "Milliseconds",

        "Dimensions": [

            {
                "Name":
                    "Version",

                "Value":
                    "V6"
            }
        ]
    },

    {
        "MetricName":
            "AverageCost",

        "Value":
            float(v6_cost),

        "Unit":
            "None",

        "Dimensions": [

            {
                "Name":
                    "Version",

                "Value":
                    "V6"
            }
        ]
    },

    {
        "MetricName":
            "ExplorationRate",

        "Value":
            float(v6_exploration_rate),

        "Unit":
            "None",

        "Dimensions": [

            {
                "Name":
                    "Version",

                "Value":
                    "V6"
            }
        ]
    },

    {
        "MetricName":
            "DynamoDBStateUpdates",

        "Value":
            float(
                len(v6_results_df)
            ),

        "Unit":
            "Count",

        "Dimensions": [

            {
                "Name":
                    "Version",

                "Value":
                    "V6"
            }
        ]
    },

    {
        "MetricName":
            "RestartRecovery",

        "Value":
            float(
                int(
                    restart_recovered
                )
            ),

        "Unit":
            "Count",

        "Dimensions": [

            {
                "Name":
                    "Version",

                "Value":
                    "V6"
            }
        ]
    }
]


try:

    cloudwatch.put_metric_data(

        Namespace=
            "AdaptiveLLMRouter/V6",

        MetricData=
            metric_data
    )

    print(
        "\nCloudWatch metrics published successfully."
    )

except ClientError as e:

    print(
        "\nCloudWatch metric publishing failed:"
    )

    print(e)


# ============================================================
# 25. FINAL STATUS
# ============================================================

print("\n")
print("=" * 100)
print("V6 COMPLETE")
print("=" * 100)

print(
    "\nDynamoDB table:"
)

print(
    DDB_TABLE_NAME
)


print(
    "\nOnline requests:"
)

print(
    ONLINE_REQUESTS
)


print(
    "\nV5 average quality:",
    round(
        v5_quality,
        4
    )
)

print(
    "V6 average quality:",
    round(
        v6_quality,
        4
    )
)


print(
    "\nV5 average latency:",
    round(
        v5_latency,
        2
    ),
    "ms"
)

print(
    "V6 average latency:",
    round(
        v6_latency,
        2
    ),
    "ms"
)


print(
    "\nV5 average cost:",
    round(
        v5_cost,
        4
    )
)

print(
    "V6 average cost:",
    round(
        v6_cost,
        4
    )
)


print(
    "\nV5 average reward:",
    round(
        v5_reward,
        4
    )
)

print(
    "V6 average reward:",
    round(
        v6_reward,
        4
    )
)


print(
    "\nV6 exploration rate:",
    f"{v6_exploration_rate:.2%}"
)

print(
    "V6 final epsilon:",
    f"{v6_final_epsilon:.4f}"
)


print(
    "\nDynamoDB restart recovery:"
)

print(
    "SUCCESS"
    if restart_recovered
    else "FAILED"
)


print(
    "\nS3 V6:"
)

print(
    f"s3://{bucket_name}/"
    f"{S3_V6_PREFIX}/"
)


print(
    "\nCloudWatch:"
)

print(
    "AdaptiveLLMRouter/V6"
)


print(
    "\nS3 upload failures:"
)

print(
    upload_failures
    if upload_failures
    else "None"
)


print(
    "\nLocal V6 artifacts:"
)

for filename in v6_artifacts:

    print(
        os.path.join(
            V6_DIR,
            filename
        )
    )


print(
    "\nV6 online adaptation is complete."
)

V6 — FINAL ONLINE ADAPTIVE LLM ROUTER

AWS region:
eu-north-1

AWS account:
753176172779

S3 bucket:
adaptive-llm-router-753176172779-eu-north-1

DynamoDB table:
adaptive-llm-router-v6-state-753176172779

Downloaded V5 state from S3:
s3://adaptive-llm-router-753176172779-eu-north-1/adaptive-llm-router/v5/v5_bandit_state.csv

V5 state records: 9

DynamoDB table already exists.

V6 DynamoDB state initialized from V5.


ONLINE QUERY SOURCE
Available queries: 639

Complexity distribution:
complexity
simple     213
medium     213
complex    213
Name: count, dtype: int64


V6 ONLINE STREAM
Requests: 300

Complexity distribution:
complexity
simple     100
medium     100
complex    100
Name: count, dtype: int64


V5 VS V6
            metric       V5       V6  delta_V6_minus_V5
   average_quality   0.9231   0.9217            -0.0014
average_latency_ms 638.7457 658.9728            20.2271
      average_cost   0.4745   0.4886             0.0141
      success_rate   1.0000   1.0000             0.0

In [ ]:
# ============================================================
# V6 — INTERACTIVE PROMPT ROUTING TEST
# ============================================================
#
# Tests completely new prompts against the learned V6 state.
#
# IMPORTANT:
# V6 currently expects:
#     prompt + complexity context
#
# This tester does NOT update DynamoDB.
# It only reads the learned state and selects a model.
# ============================================================

import boto3
import numpy as np
import pandas as pd
from botocore.exceptions import ClientError


# ============================================================
# 1. AWS CONFIG
# ============================================================

session = boto3.Session()

region = session.region_name

if region is None:
    raise RuntimeError("AWS region could not be determined.")

sts = session.client("sts")

account_id = sts.get_caller_identity()["Account"]

dynamodb = session.resource(
    "dynamodb",
    region_name=region
)

TABLE_NAME = (
    f"adaptive-llm-router-v6-state-{account_id}"
)

table = dynamodb.Table(TABLE_NAME)


MODELS = [
    "small_model",
    "medium_model",
    "large_model"
]

COMPLEXITIES = [
    "simple",
    "medium",
    "complex"
]

RNG = np.random.default_rng(12345)


# ============================================================
# 2. READ CURRENT V6 STATE
# ============================================================

def get_state(complexity, model):

    key = f"STATE#{complexity}#{model}"

    response = table.get_item(
        Key={
            "state_key": key
        }
    )

    item = response.get("Item")

    if item is None:
        raise RuntimeError(
            f"Missing DynamoDB state for {key}"
        )

    return {
        "reward": float(item["estimated_reward"]),
        "count": int(item["count"])
    }


def get_epsilon():

    response = table.get_item(
        Key={
            "state_key": "CONFIG"
        }
    )

    item = response.get("Item")

    if item is None:
        return 0.05

    return float(item["epsilon"])


# ============================================================
# 3. SELECT MODEL
# ============================================================

def select_model(complexity):

    if complexity not in COMPLEXITIES:
        raise ValueError(
            "Complexity must be simple, medium, or complex."
        )

    epsilon = get_epsilon()

    estimates = {}

    counts = {}

    for model in MODELS:

        state = get_state(
            complexity,
            model
        )

        estimates[model] = state["reward"]
        counts[model] = state["count"]

    # Exploration
    if RNG.random() < epsilon:

        selected_model = RNG.choice(
            MODELS
        )

        exploration = True

    # Exploitation
    else:

        best_reward = max(
            estimates.values()
        )

        best_models = [
            model
            for model, reward
            in estimates.items()
            if reward == best_reward
        ]

        selected_model = RNG.choice(
            best_models
        )

        exploration = False

    return (
        selected_model,
        exploration,
        epsilon,
        estimates,
        counts
    )


# ============================================================
# 4. TEST PROMPTS
# ============================================================

test_prompts = [

    {
        "prompt":
            "What is Python?",
        "complexity":
            "simple"
    },

    {
        "prompt":
            "What is overfitting in machine learning?",
        "complexity":
            "simple"
    },

    {
        "prompt":
            "What is the capital of France?",
        "complexity":
            "simple"
    },

    {
        "prompt":
            "Explain the difference between supervised and unsupervised learning with examples.",
        "complexity":
            "medium"
    },

    {
        "prompt":
            "Compare Python and Java for backend development, considering performance, ecosystem, and ease of development.",
        "complexity":
            "medium"
    },

    {
        "prompt":
            "Explain the advantages and disadvantages of using microservices instead of a monolithic architecture.",
        "complexity":
            "medium"
    },

    {
        "prompt":
            "Design a scalable fraud detection system for millions of financial transactions and explain its architecture, model selection, failure handling, monitoring, and deployment strategy.",
        "complexity":
            "complex"
    },

    {
        "prompt":
            "Design an adaptive LLM routing system that minimizes cost and latency while maintaining quality guarantees under changing workloads.",
        "complexity":
            "complex"
    },

    {
        "prompt":
            "Design a production-grade autonomous coding agent that diagnoses GitHub issues, generates patches, verifies them, handles model failures, and continuously learns which LLM should handle each task.",
        "complexity":
            "complex"
    }
]


# ============================================================
# 5. RUN TESTS
# ============================================================

print("=" * 110)
print("V6 — NEW PROMPT ROUTING TEST")
print("=" * 110)

print(
    f"\nDynamoDB table: {TABLE_NAME}"
)

print(
    f"Current epsilon: {get_epsilon():.4f}"
)

print(
    "\nNOTE: This test is READ-ONLY. "
    "It does not modify DynamoDB."
)

results = []


for i, item in enumerate(
    test_prompts,
    start=1
):

    prompt = item["prompt"]

    complexity = item["complexity"]

    (
        selected_model,
        exploration,
        epsilon,
        estimates,
        counts
    ) = select_model(
        complexity
    )

    print("\n" + "=" * 110)

    print(
        f"TEST {i}"
    )

    print(
        "\nPrompt:"
    )

    print(
        prompt
    )

    print(
        f"\nComplexity context: {complexity}"
    )

    print(
        f"Selected model: {selected_model}"
    )

    print(
        f"Decision type: "
        f"{'EXPLORATION' if exploration else 'EXPLOITATION'}"
    )

    print(
        f"Epsilon: {epsilon:.4f}"
    )

    print(
        "\nLearned rewards:"
    )

    for model in MODELS:

        print(
            f"  {model:<15} "
            f"reward={estimates[model]:.4f} "
            f"count={counts[model]}"
        )

    results.append({

        "test":
            i,

        "prompt":
            prompt,

        "complexity":
            complexity,

        "selected_model":
            selected_model,

        "decision":
            (
                "exploration"
                if exploration
                else "exploitation"
            ),

        "epsilon":
            epsilon,

        "small_reward":
            estimates["small_model"],

        "medium_reward":
            estimates["medium_model"],

        "large_reward":
            estimates["large_model"]
    })


# ============================================================
# 6. SUMMARY TABLE
# ============================================================

results_df = pd.DataFrame(
    results
)

print("\n")
print("=" * 110)
print("ROUTING SUMMARY")
print("=" * 110)

display(
    results_df[
        [
            "test",
            "complexity",
            "selected_model",
            "decision",
            "epsilon"
        ]
    ]
)


# ============================================================
# 7. OPTIONAL — CUSTOM PROMPT TEST
# ============================================================

print("\n")
print("=" * 110)
print("CUSTOM PROMPT TEST")
print("=" * 110)

custom_prompt = input(
    "\nEnter your own prompt "
    "(press Enter to skip): "
).strip()


if custom_prompt:

    print(
        "\nChoose complexity:"
    )

    print(
        "1 = simple"
    )

    print(
        "2 = medium"
    )

    print(
        "3 = complex"
    )

    choice = input(
        "\nEnter 1, 2, or 3: "
    ).strip()

    complexity_map = {
        "1": "simple",
        "2": "medium",
        "3": "complex"
    }

    if choice not in complexity_map:

        print(
            "\nInvalid complexity."
        )

    else:

        complexity = (
            complexity_map[choice]
        )

        (
            selected_model,
            exploration,
            epsilon,
            estimates,
            counts
        ) = select_model(
            complexity
        )

        print("\n")
        print("=" * 110)

        print(
            "CUSTOM RESULT"
        )

        print(
            "=" * 110
        )

        print(
            "\nPrompt:"
        )

        print(
            custom_prompt
        )

        print(
            f"\nComplexity: {complexity}"
        )

        print(
            f"Selected model: {selected_model}"
        )

        print(
            f"Decision: "
            f"{'EXPLORATION' if exploration else 'EXPLOITATION'}"
        )

        print(
            f"Epsilon: {epsilon:.4f}"
        )

        print(
            "\nCurrent learned state:"
        )

        for model in MODELS:

            print(
                f"  {model:<15} "
                f"reward={estimates[model]:.4f} "
                f"count={counts[model]}"
            )

V6 — NEW PROMPT ROUTING TEST

DynamoDB table: adaptive-llm-router-v6-state-753176172779
Current epsilon: 0.0111

NOTE: This test is READ-ONLY. It does not modify DynamoDB.

TEST 1

Prompt:
What is Python?

Complexity context: simple
Selected model: small_model
Decision type: EXPLOITATION
Epsilon: 0.0111

Learned rewards:
  small_model     reward=0.9097 count=163
  medium_model    reward=0.8881 count=13
  large_model     reward=0.7760 count=18

TEST 2

Prompt:
What is overfitting in machine learning?

Complexity context: simple
Selected model: small_model
Decision type: EXPLOITATION
Epsilon: 0.0111

Learned rewards:
  small_model     reward=0.9097 count=163
  medium_model    reward=0.8881 count=13
  large_model     reward=0.7760 count=18

TEST 3

Prompt:
What is the capital of France?

Complexity context: simple
Selected model: small_model
Decision type: EXPLOITATION
Epsilon: 0.0111

Learned rewards:
  small_model     reward=0.9097 count=163
  medium_model    reward=0.8881 count=13
  la

,test,complexity,selected_model,decision,epsilon
0,1,simple,small_model,exploitation,0.011115
1,2,simple,small_model,exploitation,0.011115
2,3,simple,small_model,exploitation,0.011115
3,4,medium,medium_model,exploitation,0.011115
4,5,medium,medium_model,exploitation,0.011115
5,6,medium,medium_model,exploitation,0.011115
6,7,complex,large_model,exploitation,0.011115
7,8,complex,large_model,exploitation,0.011115
8,9,complex,large_model,exploitation,0.011115




CUSTOM PROMPT TEST


In [ ]:
# ============================================================
# V6 — CUSTOM PROMPT TEST
# READ-ONLY: DOES NOT MODIFY DYNAMODB
# ============================================================

import boto3
import numpy as np

REGION = "eu-north-1"
ACCOUNT_ID = "753176172779"

TABLE_NAME = f"adaptive-llm-router-v6-state-{ACCOUNT_ID}"

dynamodb = boto3.resource(
    "dynamodb",
    region_name=REGION
)

table = dynamodb.Table(TABLE_NAME)

MODELS = [
    "small_model",
    "medium_model",
    "large_model"
]

rng = np.random.default_rng()


# ------------------------------------------------------------
# READ STATE
# ------------------------------------------------------------

def get_state(complexity, model):

    response = table.get_item(
        Key={
            "state_key":
                f"STATE#{complexity}#{model}"
        }
    )

    item = response.get("Item")

    if item is None:
        raise RuntimeError(
            f"Missing state: "
            f"STATE#{complexity}#{model}"
        )

    return (
        float(item["estimated_reward"]),
        int(item["count"])
    )


def get_epsilon():

    response = table.get_item(
        Key={
            "state_key": "CONFIG"
        }
    )

    item = response.get("Item")

    if item is None:
        return 0.05

    return float(
        item.get("epsilon", 0.05)
    )


# ------------------------------------------------------------
# CUSTOM TEST LOOP
# ------------------------------------------------------------

print("=" * 100)
print("V6 — CUSTOM PROMPT TEST")
print("=" * 100)

print(f"\nDynamoDB: {TABLE_NAME}")

epsilon = get_epsilon()

print(
    f"Current epsilon: {epsilon:.4f}"
)

print(
    "\nREAD-ONLY MODE — DynamoDB will NOT be changed."
)


while True:

    print("\n" + "-" * 100)

    prompt = input(
        "\nEnter your prompt "
        "(type 'exit' to stop):\n> "
    ).strip()

    if prompt.lower() == "exit":
        break

    if not prompt:
        print("Please enter a prompt.")
        continue


    # --------------------------------------------------------
    # COMPLEXITY
    # --------------------------------------------------------

    print("\nSelect complexity:")

    print("1. simple")
    print("2. medium")
    print("3. complex")

    choice = input(
        "\nChoice [1/2/3]: "
    ).strip()

    complexity_map = {
        "1": "simple",
        "2": "medium",
        "3": "complex"
    }

    if choice not in complexity_map:

        print(
            "Invalid choice."
        )

        continue

    complexity = complexity_map[choice]


    # --------------------------------------------------------
    # LOAD LEARNED STATE
    # --------------------------------------------------------

    rewards = {}
    counts = {}

    for model in MODELS:

        reward, count = get_state(
            complexity,
            model
        )

        rewards[model] = reward
        counts[model] = count


    # --------------------------------------------------------
    # V6 DECISION
    # --------------------------------------------------------

    if rng.random() < epsilon:

        selected_model = rng.choice(
            MODELS
        )

        decision = "EXPLORATION"

    else:

        selected_model = max(
            rewards,
            key=rewards.get
        )

        decision = "EXPLOITATION"


    # --------------------------------------------------------
    # DISPLAY
    # --------------------------------------------------------

    print("\n")
    print("=" * 100)
    print("V6 ROUTING RESULT")
    print("=" * 100)

    print("\nPrompt:")
    print(prompt)

    print(
        f"\nComplexity: {complexity}"
    )

    print(
        f"Selected model: {selected_model}"
    )

    print(
        f"Decision: {decision}"
    )

    print(
        f"Epsilon: {epsilon:.4f}"
    )

    print("\nLearned rewards:")

    for model in MODELS:

        print(
            f"  {model:<15}"
            f" reward={rewards[model]:.4f}"
            f"  count={counts[model]}"
        )

    print(
        "\nExpected best model: "
        f"{max(rewards, key=rewards.get)}"
    )

    print("=" * 100)

V6 — CUSTOM PROMPT TEST

DynamoDB: adaptive-llm-router-v6-state-753176172779
Current epsilon: 0.0111

READ-ONLY MODE — DynamoDB will NOT be changed.

----------------------------------------------------------------------------------------------------

Select complexity:
1. simple
2. medium
3. complex


V6 ROUTING RESULT

Prompt:
Whats the nearest mall

Complexity: simple
Selected model: small_model
Decision: EXPLOITATION
Epsilon: 0.0111

Learned rewards:
  small_model     reward=0.9097  count=163
  medium_model    reward=0.8881  count=13
  large_model     reward=0.7760  count=18

Expected best model: small_model

----------------------------------------------------------------------------------------------------

Select complexity:
1. simple
2. medium
3. complex


In [2]:
from pathlib import Path

PROJECT_ROOT = Path("/home/sagemaker-user/adaptive-llm-routing")

for path in sorted(PROJECT_ROOT.rglob("*")):
    relative = path.relative_to(PROJECT_ROOT)

    if path.is_dir():
        print(f"[DIR]  {relative}/")
    else:
        print(f"[FILE] {relative}")
       

[FILE] .env.example
[DIR]  .github/
[DIR]  .github/workflows/
[FILE] .github/workflows/ci.yml
[FILE] .gitignore
[FILE] Dockerfile
[FILE] README.md
[DIR]  app/
[DIR]  app/agents/
[FILE] app/agents/__init__.py
[FILE] app/agents/diagnosis.py
[FILE] app/agents/issue_agent.py
[FILE] app/agents/patch_generator.py
[FILE] app/agents/verification.py
[DIR]  app/api/
[FILE] app/api/__init__.py
[FILE] app/api/routes.py
[DIR]  app/cache/
[FILE] app/cache/__init__.py
[FILE] app/cache/redis_cache.py
[DIR]  app/evaluation/
[FILE] app/evaluation/__init__.py
[DIR]  app/evaluation/__pycache__/
[FILE] app/evaluation/__pycache__/__init__.cpython-312.pyc
[FILE] app/evaluation/__pycache__/audit_labels.cpython-312.pyc
[FILE] app/evaluation/audit_labels.py
[FILE] app/evaluation/benchmark.py
[FILE] app/evaluation/evaluator.py
[FILE] app/evaluation/metrics.py
[DIR]  app/models/
[FILE] app/models/__init__.py
[FILE] app/models/bedrock.py
[FILE] app/models/registry.py
[DIR]  app/observability/
[FILE] app/observabil